# EvoHash — Baseline Evaluation

Runs all baseline attacks against each PHF and displays all primary and secondary metrics.

**Primary metrics:** ASR, Efficiency (ASR / L2), L2, Time  
**Secondary metrics:** LPIPS, Query Efficiency  

Set `N_PAIRS` to 100 for the full benchmark, or lower for a quick test.

In [ ]:
import sys
from pathlib import Path

# Make sure we can import evohash from the repo root
PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import importlib.util
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from evohash.dataset import load_image_pairs
from evohash.evaluation import (
    compute_asr,
    compute_efficiency,
    compute_lpips,
    compute_mean_l2,
    compute_mean_queries,
    image_to_array,
)
from evohash.phf import get_phf

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

DATA_DIR = PROJECT_ROOT / "data" / "imagenet_val"

# ── Config ────────────────────────────────────────────────────────────────────
N_PAIRS  = 20    # set to 100 for the full benchmark
PHFS     = ["phash", "pdq"]   # add "neuralhash" if on macOS
print(f"N_PAIRS={N_PAIRS}, PHFs={PHFS}")

In [ ]:
def load_attack(path: Path):
    """Dynamically load entrypoint() from a .py file."""
    spec = importlib.util.spec_from_file_location("attack", path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.entrypoint


def evaluate_one(attack_fn, phf_name: str, n_pairs: int) -> dict:
    """Run one attack and return all metrics."""
    phf = get_phf(phf_name)
    pairs = load_image_pairs(DATA_DIR, n_pairs=n_pairs)
    sources = [p[0] for p in pairs]
    targets = [p[1] for p in pairs]
    target_hashes = [phf.compute(img) for img in targets]

    context = {
        "hash_fn": phf,
        "threshold": phf.threshold,
        "source_images": sources,
        "target_hashes": target_hashes,
        "target_images": targets,
    }

    t0 = time.time()
    result = attack_fn(context)
    elapsed = time.time() - t0

    attacked_pil = result.get("attacked_images", [])
    per_image    = result.get("metrics", [])

    originals     = [image_to_array(img) for img in sources]
    attacked_arrs = [image_to_array(img) for img in attacked_pil]

    asr          = compute_asr(per_image)
    mean_l2      = compute_mean_l2(originals, attacked_arrs)
    efficiency   = compute_efficiency(asr, mean_l2)
    mean_queries = compute_mean_queries(per_image)
    lpips_score  = compute_lpips(originals, attacked_arrs)
    time_per     = elapsed / max(len(sources), 1)

    return {
        "asr":          asr,
        "l2":           mean_l2,
        "efficiency":   efficiency,
        "queries":      mean_queries,
        "lpips":        lpips_score,
        "time_s":       time_per,
    }

## Run all baselines

In [ ]:
rows = []

for phf_name in PHFS:
    seeds_dir = PROJECT_ROOT / "problems" / phf_name / "initial_programs"
    programs  = sorted(seeds_dir.glob("*.py"))

    print(f"\n{'='*60}")
    print(f"  PHF: {phf_name.upper()}  ({len(programs)} baselines)")
    print(f"{'='*60}")

    for prog_path in programs:
        if prog_path.stem == "placeholder":
            continue
        print(f"  [{phf_name}] {prog_path.stem} ...", end=" ", flush=True)
        try:
            fn = load_attack(prog_path)
            m  = evaluate_one(fn, phf_name, N_PAIRS)
            rows.append({"attack": prog_path.stem, "phf": phf_name, **m})
            print(f"ASR={m['asr']:.2f}  L2={m['l2']:.2f}  eff={m['efficiency']:.4f}")
        except Exception as e:
            print(f"ERROR: {e}")

df = pd.DataFrame(rows)
print(f"\nDone. {len(df)} results collected.")

## Results table

In [ ]:
def fmt(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["ASR"]        = d["asr"].map("{:.3f}".format)
    d["L2"]         = d["l2"].map("{:.2f}".format)
    d["Efficiency"] = d["efficiency"].map("{:.4f}".format)
    d["Queries"]    = d["queries"].map("{:.0f}".format)
    d["LPIPS"]      = d["lpips"].apply(lambda v: "{:.4f}".format(v) if not np.isnan(v) else "N/A")
    d["Time (s)"]   = d["time_s"].map("{:.2f}".format)
    return d[["attack", "phf", "ASR", "L2", "Efficiency", "Queries", "LPIPS", "Time (s)"]]

for phf_name in PHFS:
    sub = df[df["phf"] == phf_name].sort_values("efficiency", ascending=False)
    print(f"\n── {phf_name.upper()} ──")
    display(fmt(sub).set_index("attack").drop(columns="phf"))

## Efficiency comparison (all PHFs)

In [ ]:
fig, axes = plt.subplots(1, len(PHFS), figsize=(7 * len(PHFS), 5), sharey=False)
if len(PHFS) == 1:
    axes = [axes]

for ax, phf_name in zip(axes, PHFS):
    sub = df[df["phf"] == phf_name].sort_values("efficiency", ascending=True)
    bars = ax.barh(sub["attack"], sub["efficiency"], color=sns.color_palette()[0])
    ax.set_title(phf_name.upper(), fontsize=13, fontweight="bold")
    ax.set_xlabel("Efficiency (ASR / L2)")
    ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)

plt.suptitle("Baseline Efficiency by PHF", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## ASR vs L2 scatter

In [ ]:
fig, axes = plt.subplots(1, len(PHFS), figsize=(6 * len(PHFS), 5))
if len(PHFS) == 1:
    axes = [axes]

for ax, phf_name in zip(axes, PHFS):
    sub = df[df["phf"] == phf_name]
    sc  = ax.scatter(sub["l2"], sub["asr"], s=80, zorder=3)
    for _, row in sub.iterrows():
        ax.annotate(
            row["attack"].replace("_attack", ""),
            (row["l2"], row["asr"]),
            textcoords="offset points", xytext=(6, 3), fontsize=7,
        )
    ax.set_xlabel("Mean L2 distortion")
    ax.set_ylabel("ASR")
    ax.set_title(phf_name.upper())
    ax.set_ylim(-0.05, 1.05)

plt.suptitle("ASR vs L2 (upper-left = better)", fontsize=13)
plt.tight_layout()
plt.show()

## Heatmap: all metrics

In [ ]:
for phf_name in PHFS:
    sub = df[df["phf"] == phf_name].set_index("attack")
    metrics_cols = ["asr", "l2", "efficiency", "queries", "time_s"]
    heat = sub[metrics_cols].copy()

    # Normalise each column to [0, 1] for visual comparison
    heat_norm = (heat - heat.min()) / (heat.max() - heat.min() + 1e-9)

    fig, ax = plt.subplots(figsize=(8, 0.5 * len(heat) + 2))
    sns.heatmap(
        heat_norm, annot=heat.round(3), fmt="g",
        cmap="RdYlGn", linewidths=0.5, ax=ax,
        cbar_kws={"label": "normalised value"},
    )
    ax.set_title(f"{phf_name.upper()} — all metrics (colour = normalised)", fontsize=12)
    plt.tight_layout()
    plt.show()

## Save to CSV

In [ ]:
out_path = PROJECT_ROOT / "results_baselines.csv"
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
df